# 01 — GBM expression (TCGA RNA-seq + Affymetrix microarray)

Builds the GBM expression matrix the way the gbmMINER paper does: TCGA RNA-seq and Affymetrix microarray patients are combined so that the cohort reaches about 550 patients. This notebook is specific to GBM; the other cancers use recount3 RNA-seq only.

**Each platform is z-scored on its own before the two are combined.** Microarray intensities and RNA-seq log2 TPM are on different scales, and the offset differs from gene to gene. If the two are joined first and z-scored together (as the May 2026 run did), the largest pattern in the data is the platform: in that run PC1 separated the platforms almost perfectly and 18 of the 23 transcriptional states contained patients of a single platform.

## Inputs (in `data/gbm/gbm_data/`)

| File | Source | Format |
|---|---|---|
| `rnaseq_gdc/<file UUID>/*.tsv` | GDC, STAR gene counts | one file per sample; `gene_id`, `gene_name`, `gene_type`, `tpm_unstranded`, ... |
| `gbm_tcga/data_mrna_affymetrix_microarray.txt` | cBioPortal `gbm_tcga` | genes (`Hugo_Symbol`, `Entrez_Gene_Id`) × samples; log2 intensity |

## Outputs (in `data/gbm/gbm_data/`)

| File | Format |
|---|---|
| `gbm_log2tpm.csv` | genes (symbols) × RNA-seq patients; log2(TPM+1). Used by 06 and 07 for the TF expression filter |
| `gbm_combined_raw.csv` | genes × all patients on each platform's own scale (reference only, not used downstream) |
| `gbm_zscored.csv` | genes × all patients; z-scored within platform. **This is the matrix MINER runs on** |
| `gbm_platform.csv` | one row per patient: `rnaseq` or `microarray` |
| `gbm_gene_cross_platform_correlation.csv` | one row per gene: correlation of its microarray and RNA-seq values across the patients measured on both |

**Gene reliability filter.** About 260 patients were measured on both platforms. A gene whose two measurements disagree for the same tumours (correlation below `config.MIN_CROSS_PLATFORM_R`, default 0.3) is dropped, because z-scoring cannot remove that kind of platform noise. This step is not described in the paper.


In [1]:
# Repository root: holds config.py (paths and settings) and driver_tiers.py
import os
import sys
repo_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
import config

# GBM data folder (raw downloads and the files this notebook writes)
CANCER   = "GBM"
prefix   = CANCER.lower()
data_dir = os.path.join(config.BASE_DIR, prefix, f"{prefix}_data")

import pandas as pd
import numpy as np
import requests
from scipy import stats

all_samples = {}

# Folder of GDC RNA-seq downloads
output_folder = os.path.join(data_dir, "rnaseq_gdc")

# Expression data was downloaded as a folder of folders
# Each subfolder has a file corresponding to it. The subfolder is given a Universally Unique Identifier (UUID)

for uuid in os.listdir(output_folder):
    uuid_path = os.path.join(output_folder, uuid)           # Loops through all folders and skips anything that isn't a subfolder
    
    if not os.path.isdir(uuid_path):
        continue
    
    tsv_files = [f for f in os.listdir(uuid_path) if f.endswith(".tsv")]     # List of all tsv filepaths in subfolders
    
    if not tsv_files:
        continue
    
    tsv_path = os.path.join(uuid_path, tsv_files[0])                         # [0] as each folder has only one file i.e len(tsv_files) == 1
    
    try:

        # Reads the TSV file into a dataframe
        df = pd.read_csv(tsv_path, sep="\t", comment="#", header=0) 
        
        # Skip summary rows (N_unmapped, N_multimapping etc. Only keeps actual genes i.e those with an Ensembl ID
        df = df[df["gene_id"].str.startswith("ENSG")]
        
        # Filter to protein-coding genes and drop all NaNs as done in the paper: "Gene expression RNA-seq data were TMM-normalized and Z-scored
        # using 10,263 genes after filtering"
        df = df[df["gene_type"] == "protein_coding"]    
        df = df[["gene_name", "tpm_unstranded"]].dropna()
         
        # Set index to gene name
        df = df.set_index("gene_name")
        
        # Use FOLDER name (which is the file UUID) not filename as the sample_id
        # dict: all_samples, its keys are UUIDs (corresponding to one file and in turn one sample). Sample MAY NOT BE A PRIMARY TUMOUR
        # Hence we filter later 
        # values are gene expression series (tpm_unstranded column of df) indexed by gene symbol or gene name
        all_samples[uuid] = df["tpm_unstranded"]
        
    except Exception as e:
        print(f"Error in {uuid}: {e}")

print(f"Loaded {len(all_samples)} samples")
tpm_matrix = pd.DataFrame(all_samples)
tpm_matrix = tpm_matrix.dropna()
#print(f"Matrix shape: {tpm_matrix.shape}")
#print(tpm_matrix.head())

# Loaded 391 samples
# Matrix shape: (19962, 391), excluding NaNs and non-coding genes

# Summary: tpm_matrix is a 19962 genes * 391 samples matrix
# The cells of the matrix contain TPM expression values of each gene in each sample


Loaded 391 samples


In [2]:
# Our current z-scored and tpm matrices contain UUID
# We need to map each UUID to its respective TCGA barcode since the rest of the data works on barcodes
# To do this we call the GDC API

uuids = tpm_matrix.columns.tolist()
uuid_to_barcode = {}
uuid_to_sample = {}
batch_size = 200

for i in range(0, len(uuids), batch_size):
    batch = uuids[i:i+batch_size]
    payload = {
        "filters": {
            "op": "in",
            "content": {"field": "file_id", "value": batch}
        },
        "fields": "file_id,cases.submitter_id,cases.samples.submitter_id,cases.samples.sample_type",
        "size": len(batch)
    }
    response = requests.post("https://api.gdc.cancer.gov/files", json=payload)
    
    for hit in response.json()["data"]["hits"]:
        file_id = hit["file_id"]
        case_id = hit["cases"][0]["submitter_id"]
        samples = hit["cases"][0].get("samples", [])
        uuid_to_barcode[file_id] = case_id
        uuid_to_sample[file_id] = [(s["submitter_id"], s["sample_type"]) for s in samples]
    
    print(f"Processed {min(i+batch_size, len(uuids))}/{len(uuids)}")

print(f"\nMapped: {len(uuid_to_barcode)}")

# Check sample types
all_sample_types = set()
for v in uuid_to_sample.values():
    for sample_id, sample_type in v:
        all_sample_types.add(sample_type)
print(f"Sample types found: {all_sample_types}")

# Show a few examples
print("\nExamples:")
for uuid, samples in list(uuid_to_sample.items())[:5]:
    print(f"{uuid_to_barcode[uuid]}: {samples}")


Processed 200/391
Processed 391/391

Mapped: 391
Sample types found: {'Recurrent Tumor', 'Solid Tissue Normal', 'Primary Tumor'}

Examples:
TCGA-32-5222: [('TCGA-32-5222-01A', 'Primary Tumor')]
TCGA-12-0670: [('TCGA-12-0670-01B', 'Primary Tumor')]
TCGA-19-2629: [('TCGA-19-2629-01A', 'Primary Tumor')]
TCGA-06-0749: [('TCGA-06-0749-01A', 'Primary Tumor')]
TCGA-06-2561: [('TCGA-06-2561-01A', 'Primary Tumor')]


In [3]:
# Build mapping from UUID to TCGA barcode keeping only primary tumor samples
uuid_to_barcode_filtered = {}

for file_id, samples in uuid_to_sample.items():
    primary = [s for s in samples if s[1] == "Primary Tumor"]
    if primary:
        uuid_to_barcode_filtered[file_id] = uuid_to_barcode[file_id]

print(f"Primary tumor samples: {len(uuid_to_barcode_filtered)}")
print(f"Removed (recurrent/normal): {len(uuid_to_sample) - len(uuid_to_barcode_filtered)}")

# Rename columns, drop non-primary samples
tpm_matrix.columns = [uuid_to_barcode_filtered.get(col, None) for col in tpm_matrix.columns]
tpm_matrix = tpm_matrix.loc[:, tpm_matrix.columns.notna()]
print(f"Shape after filtering: {tpm_matrix.shape}")

# 372 primary tumour samples out of 391
# Shape after filtering is 19962 *372


Primary tumor samples: 372
Removed (recurrent/normal): 19
Shape after filtering: (19962, 372)


In [4]:
dupes = tpm_matrix.columns[tpm_matrix.columns.duplicated(keep=False)].unique().tolist()
print(f"Duplicates: {len(dupes)}")

# 86 duplicates; these are handled by keeping only the sample w/ highest median expression

if dupes:
    non_dup = tpm_matrix.loc[:, ~tpm_matrix.columns.isin(dupes)]
    best_samples = []
    for patient in dupes:
        cols = tpm_matrix.loc[:, tpm_matrix.columns == patient]
        best = cols.iloc[:, cols.median(axis=0).argmax()]
        best_samples.append(best)
    best_dup = pd.concat(best_samples, axis=1)
    tpm_matrix = pd.concat([non_dup, best_dup], axis=1)

print(f"After dedup: {tpm_matrix.shape}")


Duplicates: 86
After dedup: (19962, 284)


In [5]:
# log2(TPM + 1) of the RNA-seq patients
log_tpm_matrix = np.log2(tpm_matrix + 1)
print(f"RNA-seq log2 TPM: {log_tpm_matrix.shape}")


RNA-seq log2 TPM: (19962, 284)


In [6]:
# Now loading raw microarray data
micro = pd.read_csv(os.path.join(data_dir, "gbm_tcga", "data_mrna_affymetrix_microarray.txt"), sep="\t", index_col=0)

# Step 2: Drop Entrez column
micro = micro.drop(columns=["Entrez_Gene_Id"])

# Step 3: Strip sample suffixes to get patient_ID as column and set index to gene name to match RNA-seq data
micro.columns = ["-".join(col.split("-")[:3]) for col in micro.columns]
micro.index.name = "gene_name"

print(f"Shape: {micro.shape}")

# Step 4: Check duplicates 
dup_genes = micro.index[micro.index.duplicated(keep=False)].unique()
print(f"Duplicate genes: {len(dup_genes)}")

print(f"Largest number of rows for one gene: {micro.index.value_counts().max()}")


Shape: (12042, 528)
Duplicate genes: 12
Largest number of rows for one gene: 2


In [7]:
# Several microarray probes (and a few RNA-seq rows) can map to one gene symbol. As in the paper, rows of the same gene are
# averaged if they are correlated (>= 0.8), i.e. they measure the same signal; otherwise the more variable row is kept
micro_raw = micro

# Step 1: Collapse duplicates in microarray

dup_genes_micro = micro_raw.index[micro_raw.index.duplicated(keep=False)].unique()
print(f"Microarray duplicate genes: {len(dup_genes_micro)}")
non_dup_micro = micro_raw[~micro_raw.index.isin(dup_genes_micro)]
collapsed_micro = []

for gene in dup_genes_micro:
    probes = micro_raw.loc[gene].copy()
    row0 = probes.iloc[0]
    row1 = probes.iloc[1]
    corr_val = row0.corr(row1)
    if corr_val >= 0.8:
        result = probes.mean(axis=0)
        #print(f"{gene}: corr={corr_val:.2f} → averaged")
    else:
        result = row0 if row0.var() >= row1.var() else row1
        #print(f"{gene}: corr={corr_val:.2f} → kept highest variance probe")
    collapsed_micro.append(pd.DataFrame([result.values], index=[gene], columns=micro_raw.columns))

micro_clean = pd.concat([non_dup_micro, pd.concat(collapsed_micro)])
print(f"Microarray after collapsing: {micro_clean.shape}")
print(f"Duplicate genes remaining: {micro_clean.index.duplicated().sum()}")


# Step 2: Collapse duplicates in RNA-seq

dup_genes_rna = log_tpm_matrix.index[log_tpm_matrix.index.duplicated(keep=False)].unique()
print(f"\nRNA-seq duplicate genes: {len(dup_genes_rna)}")
non_dup_rna = log_tpm_matrix[~log_tpm_matrix.index.isin(dup_genes_rna)]
collapsed_rna = []

for gene in dup_genes_rna:
    probes = log_tpm_matrix.loc[gene].copy()
    row0 = probes.iloc[0]
    row1 = probes.iloc[1]
    corr_val = row0.corr(row1)
    if corr_val >= 0.8:
        result = probes.mean(axis=0)
        #print(f"{gene}: corr={corr_val:.2f} → averaged")
    else:
        result = row0 if row0.var() >= row1.var() else row1
        #print(f"{gene}: corr={corr_val:.2f} → kept highest variance probe")
    collapsed_rna.append(pd.DataFrame([result.values], index=[gene], columns=log_tpm_matrix.columns))

log_tpm_clean = pd.concat([non_dup_rna, pd.concat(collapsed_rna)])
print(f"RNA-seq after collapsing: {log_tpm_clean.shape}")
print(f"Duplicate genes remaining: {log_tpm_clean.index.duplicated().sum()}")

# Step 3: Filter to common genes
common_genes = micro_clean.index.intersection(log_tpm_clean.index)
print(f"\nCommon genes: {len(common_genes)}")

micro_common = micro_clean.loc[common_genes]
rnaseq_common = log_tpm_clean.loc[common_genes]

print(f"Microarray common: {micro_common.shape}")
print(f"RNA-seq common: {rnaseq_common.shape}")


Microarray duplicate genes: 12
Microarray after collapsing: (12030, 528)
Duplicate genes remaining: 0

RNA-seq duplicate genes: 24
RNA-seq after collapsing: (19938, 284)
Duplicate genes remaining: 0

Common genes: 10394
Microarray common: (10394, 528)
RNA-seq common: (10394, 284)


In [8]:
# RNA-seq log2 TPM with one row per gene symbol, all protein-coding genes: used by 06 and 07 for the TF expression filter
log_tpm_clean.to_csv(os.path.join(data_dir, f"{prefix}_log2tpm.csv"))
print(f"Saved {prefix}_log2tpm.csv: {log_tpm_clean.shape}")


Saved gbm_log2tpm.csv: (19938, 284)


In [9]:
# Combine the platforms: RNA-seq is preferred for patients measured on both
common_patients = micro_common.columns.intersection(rnaseq_common.columns)
rnaseq_only = rnaseq_common.columns.difference(micro_common.columns)
micro_only = micro_common.columns.difference(rnaseq_common.columns)

print(f"Patients in both: {len(common_patients)}")
print(f"RNA-seq only: {len(rnaseq_only)}")
print(f"Microarray only: {len(micro_only)}")
print(f"Total unique patients: {len(common_patients) + len(rnaseq_only) + len(micro_only)}")

# Genes must vary on both platforms
keep = (micro_common.std(axis=1) > 0) & (rnaseq_common.std(axis=1) > 0)
print(f"Genes with zero variance on a platform: {int((~keep).sum())}")
micro_common, rnaseq_common = micro_common.loc[keep], rnaseq_common.loc[keep]

# Gene reliability filter: for the patients measured on BOTH platforms, correlate each gene's microarray and RNA-seq values
# A gene the two platforms disagree on for the same tumours carries platform-specific noise into the combined matrix,
# which z-scoring cannot remove. Genes below config.MIN_CROSS_PLATFORM_R are dropped (not described in the paper)
a = micro_common.loc[:, common_patients]
b = rnaseq_common.loc[:, common_patients]
a = a.sub(a.mean(axis=1), axis=0)
b = b.sub(b.mean(axis=1), axis=0)
cross_r = (a * b).sum(axis=1) / np.sqrt((a ** 2).sum(axis=1) * (b ** 2).sum(axis=1))
cross_r.rename("cross_platform_r").to_csv(os.path.join(data_dir, f"{prefix}_gene_cross_platform_correlation.csv"))

reliable = cross_r >= config.MIN_CROSS_PLATFORM_R
print(f"Patients on both platforms: {len(common_patients)} | median cross-platform correlation per gene: {cross_r.median():.2f}")
print(f"Genes kept (r >= {config.MIN_CROSS_PLATFORM_R}): {int(reliable.sum())} of {len(cross_r)}")
micro_common, rnaseq_common = micro_common.loc[reliable], rnaseq_common.loc[reliable]

def zscore_genes(df):

    '''

    Z-scores every gene (row) across the samples (columns) of one platform.

    Args: df (df), genes x samples. Expression on the platform's own scale

    Returns: df, same shape. Each row has mean 0 and standard deviation 1

    '''

    return df.sub(df.mean(axis=1), axis=0).div(df.std(axis=1), axis=0)

# Z-score WITHIN each platform, using all of that platform's patients, and only then join the two
# This removes the gene-specific offset between microarray intensities and RNA-seq TPM
micro_z  = zscore_genes(micro_common)
rnaseq_z = zscore_genes(rnaseq_common)

rnaseq_patients = list(rnaseq_only) + list(common_patients)
combined_zscored = pd.concat([micro_z.loc[:, micro_only], rnaseq_z.loc[:, rnaseq_patients]], axis=1)
combined_raw     = pd.concat([micro_common.loc[:, micro_only], rnaseq_common.loc[:, rnaseq_patients]], axis=1)
platform = pd.Series(["microarray"] * len(micro_only) + ["rnaseq"] * len(rnaseq_patients), index=combined_zscored.columns, name="platform")

print(f"\nCombined z-scored shape: {combined_zscored.shape}")
print(f"Value range - Min: {combined_zscored.min().min():.2f}, Max: {combined_zscored.max().max():.2f} | NaN: {int(combined_zscored.isna().sum().sum())}")

combined_raw.to_csv(os.path.join(data_dir, f"{prefix}_combined_raw.csv"))
combined_zscored.to_csv(os.path.join(data_dir, f"{prefix}_zscored.csv"))
platform.rename_axis("patient").to_csv(os.path.join(data_dir, f"{prefix}_platform.csv"))
print(f"Saved {prefix}_combined_raw.csv, {prefix}_zscored.csv and {prefix}_platform.csv")


Patients in both: 262
RNA-seq only: 22
Microarray only: 266
Total unique patients: 550
Genes with zero variance on a platform: 15
Patients on both platforms: 262 | median cross-platform correlation per gene: 0.45
Genes kept (r >= 0): 9790 of 10379

Combined z-scored shape: (9790, 550)
Value range - Min: -11.58, Max: 20.99 | NaN: 0
Saved gbm_combined_raw.csv, gbm_zscored.csv and gbm_platform.csv


In [10]:
# Check that the platform is no longer the main pattern in the data
# For each of the first three principal components: share of variance, and how well it separates the two platforms
# (0.5 = the platforms overlap completely, 1.0 = they are perfectly separated). The joint z-score of the May 2026 run gave
# PC1 = 27% of variance with a separation of 0.97
centred = combined_zscored.values - combined_zscored.values.mean(axis=1, keepdims=True)
_, sing, vt = np.linalg.svd(centred, full_matrices=False)
share = sing ** 2 / np.sum(sing ** 2)
is_rna = (platform == "rnaseq").values

for k in range(3):
    a, b = vt[k][is_rna], vt[k][~is_rna]
    overlap = (a[:, None] > b[None, :]).mean()
    print(f"PC{k+1}: {share[k]:.1%} of variance | platform separation {max(overlap, 1 - overlap):.2f}")

# Per-sample mean by platform (should be close to each other)
print(combined_zscored.mean(axis=0).groupby(platform).describe()[["mean", "std"]])

PC1: 24.3% of variance | platform separation 0.61
PC2: 7.9% of variance | platform separation 0.51
PC3: 6.2% of variance | platform separation 0.51
                    mean       std
platform                          
microarray -5.225719e-02  0.263433
rnaseq     -1.250956e-17  0.527226
